In [1]:
!pip install /kaggle/input/datasets/chakrabhuanavdeva/mamba-ssn/wheels/causal_conv1d-1.6.0-cp312-cp312-linux_x86_64.whl
!pip install /kaggle/input/datasets/chakrabhuanavdeva/mamba-ssn/wheels/mamba_ssm-2.3.0-cp312-cp312-linux_x86_64.whl

Processing /kaggle/input/datasets/chakrabhuanavdeva/mamba-ssn/wheels/causal_conv1d-1.6.0-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/chakrabhuanavdeva/mamba-ssn/wheels/mamba_ssm-2.3.0-cp312-cp312-linux_x86_64.whl


In [2]:
from mamba_ssm import Mamba
print("✅ Mamba loaded successfully")

✅ Mamba loaded successfully


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGINGFACE_KEY")
login(token=secret_value_0)

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from datasets import load_dataset
from mamba_ssm import Mamba2
from tqdm import tqdm
import torch.amp as amp
import os
from datetime import datetime

# ================= 0. UTILITIES =================
def get_timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

# ================= 1. MODEL DEFINITION =================
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        x_normed = x * torch.rsqrt(norm + self.eps)
        return self.weight * x_normed

class OtterMambaLM(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_layer=24, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Mamba2 Blocks
        self.layers = nn.ModuleList([
            Mamba2(
                d_model=d_model, 
                d_state=64, 
                d_conv=4, 
                expand=2
            )
            for _ in range(n_layer)
        ])
        
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying
        self.lm_head.weight = self.embedding.weight
        
        # Simple positional embedding (opsional, bantu stabilisasi)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_seq_len, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        
        # Add positional embedding
        seq_len = input_ids.shape[1]
        x = x + self.pos_emb[:, :seq_len, :]
        
        for layer in self.layers:
            # Handle potential tuple return dari Mamba2
            layer_output = layer(x)
            if isinstance(layer_output, tuple):
                layer_output = layer_output[0]
            x = x + layer_output  # Residual connection
            
        x = self.norm_f(x)
        return self.lm_head(x)

# ================= 2. TOKENIZER SETUP =================
print("📚 Loading IndoBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

# Setup special tokens untuk Causal LM
if tokenizer.pad_token is None:
    tokenizer.pad_token = "[PAD]"
if tokenizer.eos_token is None:
    tokenizer.eos_token = "[SEP]"  # Pakai SEP sebagai EOS
if tokenizer.bos_token is None:
    tokenizer.bos_token = "[CLS]"  # Pakai CLS sebagai BOS

tokenizer.add_special_tokens({'pad_token': '[PAD]'})

vocab_size = len(tokenizer)
print(f"✅ Vocab Size: {vocab_size}")
print(f"✅ PAD Token ID: {tokenizer.pad_token_id}")
print(f"✅ EOS Token ID: {tokenizer.eos_token_id}")

# ================= 3. DATASET PREPARATION =================
print("📖 Loading and Pre-tokenizing Dataset...")
# Increase dataset size for better training
dataset = load_dataset("indonesian-nlp/wikipedia-id", split="train[:50000]")

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=256, 
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text"],
    num_proc=4 
)
tokenized_dataset.set_format("torch")

class WikiDataset(Dataset):
    def __init__(self, tokenized_data):
        self.data = tokenized_data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        input_ids = item["input_ids"]
        labels = input_ids.clone()
        
        # Mask padding
        labels[labels == tokenizer.pad_token_id] = -100
        return {"input_ids": input_ids, "labels": labels}

train_dataset = WikiDataset(tokenized_dataset)
train_loader = DataLoader(
    train_dataset, 
    batch_size=8, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True,
    drop_last=True  # Hindari batch terakhir yang kecil
)

📚 Loading IndoBERT tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/311M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1301683 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/144632 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
# ================= 4. MODEL & DEVICE SETUP =================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"\n🔧 Device: {device}")

model = OtterMambaLM(
    vocab_size=vocab_size, 
    d_model=768, 
    n_layer=4, 
    max_seq_len=256
).to(device=device, dtype=torch.float32)

# Compile model untuk speedup (PyTorch 2.0+)
if torch.__version__ >= "2.0.0":
    print("⚡ Compiling model with torch.compile...")
    model = torch.compile(model)

print(f"✅ Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ================= 5. TRAINING CONFIG =================
num_epochs = 3
gradient_accumulation_steps = 4  # Efektif batch size = 8 * 4 = 32
lr = 1e-4
weight_decay = 0.1
max_grad_norm = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=lr, 
    weight_decay=weight_decay,
    betas=(0.9, 0.95)  # Lebih stabil buat transformer-like model
)

# Learning rate scheduler
total_steps = len(train_loader) * num_epochs // gradient_accumulation_steps
warmup_steps = int(total_steps * 0.1)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=total_steps
)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
scaler = amp.GradScaler()

# ================= 6. CHECKPOINT UTILS =================
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

def save_checkpoint(epoch, step, loss, model, optimizer, scaler, scheduler):
    checkpoint_path = f"{checkpoint_dir}/otter_mamba_epoch{epoch}_step{step}.pth"
    torch.save({
        'epoch': epoch,
        'step': step,
        'loss': loss,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }, checkpoint_path)
    print(f"💾 Checkpoint saved: {checkpoint_path}")

def load_checkpoint(checkpoint_path, model, optimizer, scaler, scheduler):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    return checkpoint['epoch'], checkpoint['step']

# ================= 7. TRAINING LOOP =================
print(f"\n🏋️ Training for {num_epochs} epochs...")
print(f"📊 Gradient Accumulation: {gradient_accumulation_steps}")
print(f"📊 Effective Batch Size: {train_loader.batch_size * gradient_accumulation_steps}")
print(f"📊 Total Steps: {total_steps}\n")

model.train()
global_step = 0
best_loss = float('inf')

for epoch in range(num_epochs):
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        # AMP Forward
        with amp.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(input_ids)
            
            # Reshape untuk loss calculation
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            
            loss = criterion(
                shift_logits.view(-1, vocab_size), 
                shift_labels.view(-1)
            )
            
            # Scale loss untuk gradient accumulation
            loss = loss / gradient_accumulation_steps
        
        # Backward
        scaler.scale(loss).backward()
        
        # Gradient accumulation step
        if (batch_idx + 1) % gradient_accumulation_steps == 0:
            # Unscale gradients
            scaler.unscale_(optimizer)
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            
            # Optimizer step
            scaler.step(optimizer)
            scaler.update()
            
            # Scheduler step
            scheduler.step()
            
            optimizer.zero_grad()
            global_step += 1
        
        total_loss += loss.item() * gradient_accumulation_steps
        current_lr = scheduler.get_last_lr()[0]
        pbar.set_postfix({
            "loss": f"{loss.item() * gradient_accumulation_steps:.4f}",
            "lr": f"{current_lr:.2e}"
        })
    
    avg_loss = total_loss / len(train_loader)
    print(f"\n✅ Epoch {epoch+1} Complete - Avg Loss: {avg_loss:.4f}")
    
    # Save checkpoint setiap epoch
    save_checkpoint(epoch+1, global_step, avg_loss, model, optimizer, scaler, scheduler)
    
    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), f"{checkpoint_dir}/otter_mamba_best.pth")
        print(f"🏆 New best model saved! Loss: {best_loss:.4f}")

# ================= 8. FINAL SAVE =================
torch.save(model.state_dict(), "otter_mamba_final.pth")
print("\n💾 Final model saved to otter_mamba_final.pth")

# ================= 9. VRAM CHECK =================
print("\n📊 VRAM Usage:")
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**2
    reserved = torch.cuda.memory_reserved(i) / 1024**2
    print(f"   GPU {i}: {allocated:.2f} MB allocated, {reserved:.2f} MB reserved")




🔧 Device: cuda:0
⚡ Compiling model with torch.compile...
✅ Model Parameters: 38,299,936

🏋️ Training for 3 epochs...
📊 Gradient Accumulation: 4
📊 Effective Batch Size: 32
📊 Total Steps: 4687



Epoch 1/3: 100%|██████████| 6250/6250 [15:22<00:00,  6.77it/s, loss=5.7357, lr=8.43e-05]



✅ Epoch 1 Complete - Avg Loss: 44.4034
💾 Checkpoint saved: checkpoints/otter_mamba_epoch1_step1562.pth
🏆 New best model saved! Loss: 44.4034


Epoch 2/3: 100%|██████████| 6250/6250 [09:06<00:00, 11.44it/s, loss=4.8246, lr=3.02e-05]



✅ Epoch 2 Complete - Avg Loss: 5.6053
💾 Checkpoint saved: checkpoints/otter_mamba_epoch2_step3124.pth
🏆 New best model saved! Loss: 5.6053


Epoch 3/3: 100%|██████████| 6250/6250 [09:04<00:00, 11.47it/s, loss=3.8899, lr=1.39e-11]



✅ Epoch 3 Complete - Avg Loss: 4.9266
💾 Checkpoint saved: checkpoints/otter_mamba_epoch3_step4686.pth
🏆 New best model saved! Loss: 4.9266

💾 Final model saved to otter_mamba_final.pth

📊 VRAM Usage:
   GPU 0: 840.64 MB allocated, 1940.00 MB reserved
   GPU 1: 0.00 MB allocated, 0.00 MB reserved


In [6]:
# ================= 10. INFERENCE TEST (FINAL FIX) =================
print("\n🧪 Quick Inference Test (FINAL FIX)...")
model.eval()

def generate_text(prompt, max_length=100, temperature=0.9, top_k=50, repetition_penalty=1.2):
    """
    Generate text dengan anti-UNK dan anti-repetition.
    """
    input_ids = tokenizer.encode(
        prompt, 
        return_tensors="pt", 
        add_special_tokens=False
    ).to(device)
    
    prompt_length = input_ids.shape[1]
    
    with torch.no_grad():
        for step in range(max_length):
            with amp.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(input_ids)
            
            next_token_logits = logits[:, -1, :] / temperature
            
            # 🔥 Repetition penalty (kurangi loop "mel mel mel")
            if repetition_penalty > 1.0:
                for token_id in set(input_ids[0].tolist()):
                    if next_token_logits[0, token_id] < 0:
                        next_token_logits[0, token_id] *= repetition_penalty
                    else:
                        next_token_logits[0, token_id] /= repetition_penalty
            
            # Top-K filtering
            if top_k is not None:
                indices_to_remove = logits[:, -1, :] < torch.topk(logits[:, -1, :], top_k)[0][..., -1, None]
                next_token_logits[indices_to_remove] = -float('inf')
            
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Stop kalau EOS
            if next_token.item() == tokenizer.eos_token_id:
                break
            
            # ⚠️ Skip token yang kemungkinan besar jadi [UNK]
            if next_token.item() >= vocab_size:
                print(f"   ⚠️ Skipping invalid token ID: {next_token.item()}")
                continue
            
            input_ids = torch.cat([input_ids, next_token], dim=1)
    
    # Decode dengan fallback untuk [UNK]
    generated_ids = input_ids[0][prompt_length:].tolist()
    
    # Coba decode, kalau banyak [UNK] coba alternatif
    output = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Hitung berapa banyak [UNK]
    unk_count = output.count("[UNK]")
    if unk_count > len(output) // 3:  # Kalau lebih dari 33% [UNK]
        print(f"   ⚠️ Warning: {unk_count} [UNK] tokens detected")
    
    return output


# 🎯 Test dengan berbagai prompt
test_prompts = [
    "Indonesia adalah",
    "Jakarta adalah",
    "Majapahit",
    "Bahasa Indonesia"
]

print("="*60)
for prompt in test_prompts:
    generated = generate_text(prompt, max_length=60, temperature=0.9, top_k=40, repetition_penalty=1.2)
    print(f"\n📝 Prompt: {prompt}")
    print(f"✨ Output: {generated.strip()}")
print("\n" + "="*60)

model.train()


🧪 Quick Inference Test (FINAL FIX)...

📝 Prompt: Indonesia adalah
✨ Output: desa di kecamatan,, provinsi,.

📝 Prompt: Jakarta adalah
✨ Output: sebuah asteroid. ini merupakan bagian dari asteroid, yang terletak dekat dengan bumi. orbit asteroid ini tercatat sebesar 0. 041, sementara magnitudo mutlaknya adalah 8. 6.

📝 Prompt: Majapahit
✨ Output: , yang dikenal sebagai ( ).

📝 Prompt: Bahasa Indonesia
✨ Output: 



OptimizedModule(
  (_orig_mod): OtterMambaLM(
    (embedding): Embedding(30521, 768)
    (layers): ModuleList(
      (0-3): 4 x Mamba2(
        (in_proj): Linear(in_features=768, out_features=3224, bias=False)
        (conv1d): Conv1d(1664, 1664, kernel_size=(4,), stride=(1,), padding=(3,), groups=1664)
        (act): SiLU()
        (norm): RMSNorm()
        (out_proj): Linear(in_features=1536, out_features=768, bias=False)
      )
    )
    (norm_f): RMSNorm()
    (lm_head): Linear(in_features=768, out_features=30521, bias=False)
  )
)